## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation

In [7]:
%%ampl_eval
reset;

# Sets
set NUTR;        # all nutrients
set MAXREQ;      # nutrients with maximum requirement
set FOOD;        # all foods

# Parameters
param n_min {NUTR};      # minimum nutrient requirement
param cost  {FOOD};      # cost of each food
param f_min {FOOD};      # minimum allowed purchase
param f_max {FOOD};      # maximum allowed purchase

# Decision variable
var Buy {FOOD} >= 0;

# Filtered derived set: nutrients in MAXREQ OR with positive minimum requirement
set NUTREQ = {i in NUTR: i in MAXREQ or n_min[i] > 0};

# Filtered cost term: cost of foods with positive minimum purchase
param excess_cost =
    sum {j in FOOD: f_min[j] > 0} cost[j] * f_min[j];

# Filtered constraint: tight range forces fixed decision
subject to NarrowRange {j in FOOD: f_max[j] - f_min[j] < 1}:
    Buy[j] = f_min[j];

In [8]:
%%ampl_eval
data;

set NUTR := protein fat fiber vitamin;

set MAXREQ := protein vitamin;

set FOOD := bread milk apple meat;

param n_min :=
protein 50
fat 20
fiber 0
vitamin 10;

param: cost  f_min  f_max :=
bread   2     1      1.5
milk    3     0      2
apple   1     2      2.5
meat    5     1      5;

display NUTREQ, excess_cost, NarrowRange;

set NUTREQ := protein fat vitamin;

excess_cost = 9

NarrowRange [*] :=
apple  0
bread  0
;



### amplpy Formulation
#### AMPL Model

In [11]:
%%ampl_eval
reset;

# Sets
set NUTR;        # all nutrients
set MAXREQ;      # nutrients with maximum requirement
set FOOD;        # all foods

# Parameters
param n_min {NUTR};      # minimum nutrient requirement
param cost  {FOOD};      # cost of each food
param f_min {FOOD};      # minimum allowed purchase
param f_max {FOOD};      # maximum allowed purchase

# Decision variable
var Buy {FOOD} >= 0;

# Filtered derived set: nutrients in MAXREQ OR with positive minimum requirement
set NUTREQ = {i in NUTR: i in MAXREQ or n_min[i] > 0};

# Filtered cost term: cost of foods with positive minimum purchase
param excess_cost =
    sum {j in FOOD: f_min[j] > 0} cost[j] * f_min[j];

# Filtered constraint: tight range forces fixed decision
subject to NarrowRange {j in FOOD: f_max[j] - f_min[j] < 1}:
    Buy[j] = f_min[j];

#### Python Data

In [12]:
# Sets
ampl.set["NUTR"].set_values(["protein", "fat", "fiber", "vitamin"])
ampl.set["MAXREQ"].set_values(["protein", "vitamin"])
ampl.set["FOOD"].set_values(["bread", "milk", "apple", "meat"])

# Parameters (indexed)
ampl.param["n_min"].set_values({
    "protein": 50,
    "fat": 20,
    "fiber": 0,
    "vitamin": 10})

# Table-style parameters (aligned by FOOD index)
import pandas as pd

df = pd.DataFrame({
    "cost":  [2, 3, 1, 5],
    "f_min": [1, 0, 2, 1],
    "f_max": [1.5, 2, 2.5, 5]
}, index=["bread", "milk", "apple", "meat"])

for col in df.columns:
    ampl.param[col].set_values(df[col])


# Display results
ampl.eval("display NUTREQ, excess_cost;")

set NUTREQ := protein fat vitamin;

excess_cost = 9

